# 02 — Clause Segmentation
This notebook explores how ClauseGuard splits a full contract into individual clauses.

**Two strategies are used:**
1. **Regex-based** — detect numbered/markdown section headers (e.g. `## 1. Services`)
2. **spaCy sentence boundary detection** — fallback if no numbered structure is found

**Production file:** `backend/segmenter.py`

In [ ]:
import sys
sys.path.append('..')

import re
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded:", nlp.meta['name'])

## 1. Load sample contract text

In [ ]:
from backend.extractor import extract_text
from backend.cleaner import clean_text

raw = extract_text("../tests/sample_contract.pdf")
cleaned = clean_text(raw)
print("Cleaned contract text:")
print(cleaned[:800])

## 2. Strategy A — Numbered / Markdown Header Detection

In [ ]:
def detect_numbered_clauses(text):
    """
    Detects clause boundaries using patterns like:
      - ## 1. Services
      - Section 1. Payment
      - Article 2. Termination
    Preserves the header as part of the clause text.
    """
    pattern = r'(?:\n|^)\s*(?:##\s*|Section\s+|Article\s+)?(\d+\.\s+)'
    matches = list(re.finditer(pattern, text))
    if not matches:
        return []

    clauses = []
    preamble = text[:matches[0].start()].strip()
    if preamble:
        clauses.append(preamble)

    for i, match in enumerate(matches):
        start = match.start()
        if match.group(0).startswith('\n'):
            start += 1
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        clause_content = text[start:end].strip()
        if clause_content:
            clauses.append(clause_content)

    return clauses

clauses = detect_numbered_clauses(cleaned)
print(f"Found {len(clauses)} numbered clauses\n")
for i, c in enumerate(clauses[:5], 1):
    print(f"--- Clause {i} ---")
    print(c[:200])
    print()

## 3. Strategy B — spaCy Sentence Boundary Detection (fallback)

In [ ]:
def split_into_sentences(text):
    """Use spaCy's statistical model to split text into sentences."""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

# Demo on a plain, unnumbered text block
unnumbered_text = """
The vendor agrees to provide services as described. Payment shall be made within 30 days.
All intellectual property shall transfer to the client upon full payment.
Either party may terminate this agreement with 30 days notice.
"""

sentences = split_into_sentences(unnumbered_text)
print(f"spaCy found {len(sentences)} sentences:")
for i, s in enumerate(sentences, 1):
    print(f"  {i}. {s}")

## 4. Combined `segment_into_clauses()` — the production logic

In [ ]:
def segment_into_clauses(text):
    """Use numbered clause detection first; fall back to spaCy if no structure is found."""
    numbered = detect_numbered_clauses(text)
    if len(numbered) > 1:
        print("Strategy: Numbered/markdown clause detection")
        return numbered
    else:
        print("Strategy: spaCy sentence boundary detection (fallback)")
        return split_into_sentences(text)

final_clauses = segment_into_clauses(cleaned)
print(f"\nTotal clauses: {len(final_clauses)}")
for i, c in enumerate(final_clauses, 1):
    print(f"\n--- Clause {i} ---")
    print(c[:300])

## 5. Why not just split on newlines or periods?

| Approach | Problem |
|---|---|
| Split on `\n` | Loses multi-line clauses; treats every line as a separate clause |
| Split on `.` | Breaks on abbreviations (`Ltd.`, `Jan. 1`) and numbered lists |
| spaCy sentences | Good for prose, but misses legal section boundaries |
| **Header regex** | ✅ Best for structured contracts with numbered sections |

The hybrid approach handles both structured (numbered) and unstructured (plain prose) contracts.